In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
future_before_df = pd.read_csv("../data/before_future_train_v3.csv", index_col=0)
future_train_df = pd.read_csv("../data/future_train_v3.csv", index_col=0)
future_test_df = pd.read_csv("../data/future_test_v3.csv", index_col=0)
future_tot_train_df = pd.concat([future_before_df, future_train_df], axis=0)
future_tot_train_df.reset_index(drop=True, inplace=True)
future_tot_train_df = pd.concat([future_tot_train_df, future_test_df], axis=0)
future_tot_train_df.reset_index(drop=True, inplace=True)
future_tot_train_df.sort_values(by=['date', 'tic_code'], inplace=True)
future_tot_train_df.reset_index(drop=True, inplace=True)

In [3]:
before_df = pd.read_csv("../data/before_train_v3.csv", index_col=0)
train_df = pd.read_csv("../data/train_v3.csv", index_col=0)
test_df = pd.read_csv("../data/test_v3.csv", index_col=0)
tot_train_df = pd.concat([before_df, train_df], axis=0)
tot_train_df.reset_index(drop=True, inplace=True)
tot_train_df = pd.concat([tot_train_df, test_df], axis=0)
tot_train_df.reset_index(drop=True, inplace=True)
tot_train_df.sort_values(by=['date', 'tic_code'], inplace=True)
tot_train_df.reset_index(drop=True, inplace=True)

In [4]:
selected_feat = ['date', 'tic_code', 'close', 'ticker',
       'ret', 'high_ratio', 'low_ratio', 'adjust_close', 'adjust_high',
       'adjust_low', 'return_1m', 'return_3m', 'return_6m', 'return_12m',
       'return_avg', 'mom_12m', 'mom_score', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'SMA_30', 'SMA_60', 'SMA_220',
       'SMA_252', 'macd', 'boll_ub', 'boll_lb', 'rsi', 'cci', 'paa_score',
       'gtaa_buy_signal', 'paa_buy_signal', 'daa_buy_signal',
       'momentum_product', 'dm_buy_signal', 'adjust_close_norm',
       'return_1m_norm', 'return_3m_norm', 'return_6m_norm', 'return_12m_norm',
       'return_avg_norm', 'mom_12m_norm', 'mom_score_norm', 'SMA_30_norm',
       'SMA_60_norm', 'SMA_220_norm', 'SMA_252_norm', 'macd_norm',
       'boll_ub_norm', 'boll_lb_norm', 'rsi_norm', 'cci_norm']

In [5]:
eqaul_df = pd.concat([tot_train_df[selected_feat], future_tot_train_df[selected_feat]])

In [6]:
eqaul_df.sort_values(by=['date', 'tic_code'], inplace=True)
eqaul_df.reset_index(drop=True, inplace=True)

In [7]:
# TRAIN_START_DATE = '2002-02-05'
# TRAIN_END_DATE = '2012-12-31'
# VAILD_START_DATE = '2013-01-01'
# VAILD_END_DATE = '2018-12-31'
# TEST_START_DATE = '2019-01-01'
# TEST_END_DATE = '2024-12-31'

In [7]:
TRAIN_START_DATE = '2002-02-05'
TRAIN_END_DATE = '2012-12-31'

TRADE_START_DATE = '2013-01-01'
TRADE_END_DATE = '2024-12-31'

# TRAIN_START_DATE = '2000-01-10'
# TRAIN_END_DATE = '2023-12-31'
# TRADE_START_DATE = '2010-01-01'
# TRADE_END_DATE = '2023-12-31'

window_days = 252
start_day = 252
top_k = 5
top_pct = 0.5
risk_free_rate = 0
gamma = 10
rebalance_every = 5
cost = 0.003
annual_factor = 252

In [8]:
future_price = future_tot_train_df[["date", "ticker", "close"]].dropna()
index_price = tot_train_df[["date", "ticker", "close"]].dropna()
equal_price = eqaul_df[["date", "ticker", "close"]].dropna()

In [9]:
future_df = future_price.pivot(index="date", columns="ticker", values="close")
index_df = index_price.pivot(index="date", columns="ticker", values="close")
equal_df = equal_price.pivot(index="date", columns="ticker", values="close")

In [10]:
future_df = future_df.sort_index()
index_df = index_df.sort_index()
equal_df = equal_df.sort_index()

In [11]:
future_return = future_df.pct_change().fillna(0)
index_return = index_df.pct_change().fillna(0)
equal_return = equal_df.pct_change().fillna(0)

In [12]:
# ✅ 평균-분산 (Mean-Variance), 최소분산 (MinVar), 리스크패리티 (Risk-Parity), Max Sharpe
# 전략들의 차이는 "최적화 기준"의 차이뿐이며, 백테스트 로직 구조는 거의 동일함

import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt

# ✅ 평균-분산 최적화 함수 (공매도 금지)

# ✅ 최소분산 포트폴리오 (Min-Var)
def compute_minvar_weights(mu, cov_matrix):
    n = cov_matrix.shape[0]
    w = cp.Variable(n)
    objective = cp.Minimize(cp.quad_form(w, cov_matrix))
    constraints = [cp.sum(w) == 1, w >= 0]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    return w.value


# ✅ 리스크 패리티 가중치 (역분산 기반 근사)
def compute_risk_parity_weight_from_window(returns_window):
    var = returns_window.var()
    inv_var = 1 / var
    weights = inv_var / inv_var.sum()
    return weights


def compute_max_sharpe_min_var(mu, cov_matrix, risk_free_rate=0.0, max_vol=None, target_return=0):
    n = len(mu)
    x = cp.Variable(n)
    excess_mu = mu - risk_free_rate
    # objective = cp.Minimize(cp.quad_form(x, cov_matrix))
    w_tilde = cp.Variable(n)  # 치환된 weight (w_tilde = k * w)
    k = cp.Variable(nonneg=True)  # 스케일 변수
    objective = cp.Minimize(cp.quad_form(w_tilde, cov_matrix))

    # constraints = [cp.sum(x) == 1, x >= 0, excess_mu @ x >= target_return]
    # constraints = [cp.sum(x) == 1, x >= 0, excess_mu @ x == 1]
    constraints = [
        excess_mu.T @ w_tilde == 1,   # 초과수익률 고정
        cp.sum(w_tilde) == k,          # w_tilde = k * w
        w_tilde >= 0          # w_tilde = k * w
    ]


    prob = cp.Problem(objective, constraints)
    prob.solve()
    
    # try:
    #     prob.solve()
    #     if x.value is not None:
    #         return x.value
    #     else:
    #         raise ValueError("No solution from solver")
    # except Exception as e:
    #     return np.ones(n) / n  # fallback: equal weights
    if w_tilde.value is not None and k.value is not None and k.value > 0:
        w = w_tilde.value / k.value
        return w
    else:
        print("최적화 실패. 균등 포트폴리오로 fallback.")
        return np.zeros(n)

# ✅ 백테스트 공통 함수 (최적화 함수 인자로 받음)
def backtest_strategy(returns, compute_weights_fn, rebalance_every=20, window_days=252, cost=0.003, start_day=252, **kwargs):
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = None

    i = start_day
    while i < len(dates) - rebalance_every:
        window_data = returns.iloc[i - window_days:i]
        # print(f"Window data shape: {window_days}")
        if window_data.isna().sum().sum() > 0:
            i += rebalance_every
            continue

        mu = window_data.mean().values
        cov = window_data.cov().values

        # mu = window_data.mean().values
        # cov = window_data.cov().values
        try:
            weights = compute_weights_fn(mu, cov, **kwargs)
        except TypeError:
            weights = compute_weights_fn(window_data, **kwargs)

        if weights is None:
            i += rebalance_every
            continue

        n_assets = returns.shape[1]
        
        if prev_weights is None:
            prev_weights = np.zeros(n_assets)

        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
            
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            if j == 0 and prev_weights is not None:
                turnover = np.abs(weights - prev_weights).sum()
                tc = turnover * cost
            else:
                tc = 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights))

        prev_weights = weights
        i += rebalance_every

    strat_returns = pd.Series(dict(portfolio_returns)).sort_index()
    strat_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return strat_returns, strat_weights


## 다른 전략

In [13]:
# 3. 리스크 패리티 (Risk-Parity)
future_rp_returns, future_rp_weights = backtest_strategy(
    future_return,
    compute_weights_fn=compute_risk_parity_weight_from_window,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost=cost
)

index_rp_returns, index_rp_weights = backtest_strategy(
    index_return,
    compute_weights_fn=compute_risk_parity_weight_from_window,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost = cost
)

In [14]:
index_rp_returns

2002-02-05   -0.015220
2002-02-06   -0.005098
2002-02-07    0.004381
2002-02-08    0.004523
2002-02-11    0.010565
                ...   
2024-12-24    0.004831
2024-12-26    0.002980
2024-12-27   -0.002778
2024-12-30   -0.003768
2024-12-31   -0.000138
Length: 5731, dtype: float64

In [15]:
future_minvar_returns, future_minvar_weights = backtest_strategy(
    future_return,
    compute_weights_fn=compute_minvar_weights,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost=cost
)

future_minvar_weights.columns = future_return.columns

index_minvar_returns, index_minvar_weights = backtest_strategy(
    index_return,
    compute_weights_fn=compute_minvar_weights,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost=cost
)
index_minvar_weights.columns = index_return.columns

In [16]:
future_ms_returns, future_ms_weights = backtest_strategy(
    future_return,
    compute_weights_fn=compute_max_sharpe_min_var,
    risk_free_rate=risk_free_rate,  # 무위험 수익률
    target_return=0.0,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost =cost


    # max_vol=0.01   # 최대 변동성 제약 (선형 근사)
)
future_ms_weights.columns = future_return.columns

/opt/conda/lib/python3.11/site-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.


In [17]:
index_ms_returns, index_ms_weights = backtest_strategy(
    index_return,
    compute_weights_fn=compute_max_sharpe_min_var,
    risk_free_rate=risk_free_rate,  # 무위험 수익률
    target_return=0.0,
    rebalance_every=rebalance_every,
    window_days=window_days,
    start_day = start_day,
    cost = cost


    # max_vol=0.01   # 최대 변동성 제약 (선형 근사)
)
index_ms_weights.columns = index_return.columns

/opt/conda/lib/python3.11/site-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fallback.
최적화 실패. 균등 포트폴리오로 fa

In [18]:
def backtest_paa_from_pivot(returns: pd.DataFrame,
                             pivot_score: pd.DataFrame,
                             window_days: int = 252,
                             rebalance_every: int = 20,
                             top_n: int = 10,
                             score_threshold: float = 0.0,
                             cost: float = 0.003,
                             start_day: int = 252):
    """
    PAA 전략 백테스트 (수익률 기반 점수 사용, 거래비용 반영)

    Parameters:
    - returns: 일간 수익률 DataFrame (index: date, columns: tickers)
    - pivot_score: PAA score DataFrame (index: date, columns: tickers)
    - window_days: 과거 데이터 시작 시점
    - rebalance_every: 리밸런싱 주기
    - top_n: 상위 자산 수
    - score_threshold: 점수 필터링 기준
    - cost: 거래 비용 비율

    Returns:
    - paa_returns: 전략 일간 수익률 Series
    - paa_weights: 전략 리밸런싱 시점별 자산 비중 DataFrame
    """
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = pd.Series(0, index=returns.columns, dtype=float)

    i = start_day
    while i < len(dates) - rebalance_every:
        current_date = dates[i]
        score_today = pivot_score.loc[current_date]
        satisfied_assets = score_today[score_today > score_threshold]
        if satisfied_assets.empty:
            selected = []
        else:
            selected = satisfied_assets.sort_values(ascending=False).head(top_n).index

        weights = pd.Series(0, index=returns.columns, dtype=float)
        if len(selected) > 0:
            weights[selected] = 1 / len(selected)

        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]
                    
        for j, (date, row) in enumerate(sub_returns.iterrows()):
            tc = np.abs(weights - prev_weights).sum() * cost if j == 0 else 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights.copy()))

        prev_weights = weights.copy()
        i += rebalance_every

    paa_returns = pd.Series(dict(portfolio_returns)).sort_index()
    paa_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return paa_returns, paa_weights

In [19]:
future_paa_score = future_tot_train_df.pivot(index='date', columns='ticker', values='paa_score')

In [20]:
future_paa_returns, future_paa_weights = backtest_paa_from_pivot(
    future_return, future_paa_score,
    top_n=top_k,
    window_days = window_days,
    rebalance_every=rebalance_every,
    score_threshold=0.0,
    cost=cost
)

In [21]:
index_paa_score = tot_train_df.pivot(index='date', columns='ticker', values='paa_score')


In [22]:
index_paa_returns, index_paa_weights = backtest_paa_from_pivot(
    index_return, index_paa_score,
    top_n=top_k,
    window_days = window_days,
    rebalance_every=rebalance_every,
    score_threshold=0.0,
    cost=cost
)

In [23]:
def backtest_equal_weight_20day(returns: pd.DataFrame, rebalance_every=20, window_days=252, cost=0.003, start_day=252):
    dates = returns.index
    portfolio_returns = []
    weights_record = []
    prev_weights = None

    i = start_day
    while i < len(dates) - rebalance_every:
        rebalance_day = dates[i]
        window_data = returns.iloc[i - window_days:i]


        n_assets = window_data.shape[1]
        weights = np.ones(n_assets) / n_assets  # 1/N 포트폴리오
        n_assets = returns.shape[1]
        if prev_weights is None:
            prev_weights = np.zeros(n_assets)  # 최초 리밸런싱: 전량 매수로 간주
            
        if i + rebalance_every < len(dates) - rebalance_every:
            sub_returns = returns.iloc[i:i + rebalance_every]
        else:
            sub_returns = returns.iloc[i:]

        for j, (date, row) in enumerate(sub_returns.iterrows()):
            if j == 0 and prev_weights is not None:
                turnover = np.abs(weights - prev_weights).sum()
                tc = turnover * cost
            else:
                tc = 0
            daily_ret = np.dot(weights, row.fillna(0)) - tc
            portfolio_returns.append((date, daily_ret))
            weights_record.append((date, weights))

        prev_weights = weights
        i += rebalance_every

    ew_returns = pd.Series(dict(portfolio_returns)).sort_index()
    ew_weights = pd.DataFrame({d: w for d, w in weights_record}).T
    return ew_returns, ew_weights


In [24]:
# 백테스트 실행
equal_returns, equal_weights = backtest_equal_weight_20day(equal_return, rebalance_every=rebalance_every, window_days=window_days, cost=cost)

# Sharpe 비율 계산
# equal_sharpe = equal_returns.mean() / equal_returns.std() * np.sqrt(252)

In [25]:
strategy_returns = {
    'future_risk_parity': future_rp_returns,
    'index_risk_parity': index_rp_returns,
    'future_minvar': future_minvar_returns,
    'index_minvar': index_minvar_returns,
    'future_ms': future_ms_returns,
    'index_ms': index_ms_returns,
    'future_paa' : future_paa_returns,
    'index_paa' : index_paa_returns,
    'Equal Weight': equal_returns
}

In [26]:
future_risk_cum = (1 + future_rp_returns).cumprod()
index_risk_cum = (1 + index_rp_returns).cumprod()
future_minvar_cum = (1 + future_minvar_returns).cumprod()
index_minvar_cum = (1 + index_minvar_returns).cumprod()
future_ms_cum = (1 + future_ms_returns).cumprod()
index_ms_cum = (1 + index_ms_returns).cumprod()
future_paa_cum = (1 + future_paa_returns).cumprod()
index_paa_cum = (1 + index_paa_returns).cumprod()
equal_cum = (1 + equal_returns).cumprod()

In [27]:
import os
import sys
# 현재 경로의 부모 디렉토리
parent_dir = os.path.dirname(os.getcwd())

# sys.path에 현재 경로와 부모 경로 추가
sys.path.append(parent_dir)
import eval_metric as em

In [28]:
def calculate_rolling_metrics(
    returns: pd.Series,
    windows: list = [20, 252],
    risk_free_rate: float = 0.02,
    annual_factor: int = 252,
    name : str = "returns",
    shift_features: bool = True,  # ✅ 추가

) -> pd.DataFrame:
    """
    주어진 daily return 시리즈에 대해 rolling window 기반 
    Sharpe, Volatility, Sortino, Calmar 계산

    Args:
        returns (pd.Series): 일간 수익률
        windows (list): rolling window list (e.g., [20, 252])
        risk_free_rate (float): 무위험 수익률 (연환산)
        annual_factor (int): 연환산 계수 (default 252)

    Returns:
        pd.DataFrame: 원본 daily_return + 모든 rolling metric columns
    """
    result = pd.DataFrame({'daily_return': returns})
    result["tic"] = name

    for window in windows:
        result[f'sharpe_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_sharpe_ratio(x, risk_free_rate, annual_factor),
            raw=False
        )
        result[f'vol_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_volatility(x, annual_factor),
            raw=False
        )
        result[f'sortino_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_annualized_sortino_ratio(x, risk_free_rate, annual_factor),
            raw=False
        )
        result[f'calmar_{window}'] = returns.rolling(window).apply(
            lambda x: em.calculate_calmar_ratio(x, annual_factor),
            raw=False
        )
    if shift_features:
        feature_cols = [col for col in result.columns if col not in ['daily_return', 'tic']]
        result[feature_cols] = result[feature_cols].shift(1)
        
    return result

In [30]:
future_rp_eval = calculate_rolling_metrics(future_rp_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_rp")
index_rp_eval = calculate_rolling_metrics(index_rp_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="index_rp")
future_minvar_eval = calculate_rolling_metrics(future_minvar_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_minvar")
index_minvar_eval = calculate_rolling_metrics(index_minvar_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="index_minvar")
future_ms = calculate_rolling_metrics(future_ms_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_ms")
index_ms = calculate_rolling_metrics(index_ms_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="index_ms")
future_paa = calculate_rolling_metrics(future_paa_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="future_paa")
index_paa = calculate_rolling_metrics(index_paa_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="index_paa")
equal_eval = calculate_rolling_metrics(equal_returns, windows=[20, 252], risk_free_rate=risk_free_rate, annual_factor=annual_factor, name="equal")

In [33]:
future_risk_df = pd.DataFrame({
    'close' : future_risk_cum,
    "tic" : "future_risk"})

index_risk_df = pd.DataFrame({
    'close' : index_risk_cum,
    "tic" : "index_risk"})
future_minvar_df = pd.DataFrame({
    'close' : future_minvar_cum,
    "tic" : "future_minvar"})
index_minvar_df = pd.DataFrame({
    'close' : index_minvar_cum,
    "tic" : "index_minvar"})
future_ms_df = pd.DataFrame({
    'close' : future_ms_cum,
    "tic" : "future_ms"})
index_ms_df = pd.DataFrame({
    'close' : index_ms_cum,
    "tic" : "index_ms"})
future_paa_df = pd.DataFrame({
    'close' : future_paa_cum,
    "tic" : "future_paa"})
index_paa_df = pd.DataFrame({
    'close' : index_paa_cum,
    "tic" : "index_paa"})

In [34]:
import pandas as pd

def calculate_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    주어진 데이터프레임에 대해 모멘텀 관련 특성 생성.
    
    Args:
        df (pd.DataFrame): 종가(close) 컬럼을 포함한 데이터프레임
        close_col (str): 종가 컬럼 이름 (default: 'close')
        
    Returns:
        pd.DataFrame: 모멘텀 특성이 추가된 데이터프레임
    """
    data = df.copy()
    
    # 설정할 period 리스트
    periods = [20, 40, 60, 80, 100, 120, 140, 160, 180, 220, 240, 252]
    columns_list = ["return_1m", "return_3m", "return_6m", "return_12m", "return_avg", "mom_12m", "mom_score"]

    # 기간별 이전 종가 저장
    for idx, period in enumerate(periods):
        col_name = f'close_{idx+1}_month'
        data[col_name] = data['close'].shift(period)
        columns_list.append(col_name)
    
    # 모멘텀 및 수익률 계산
    data['return_1m'] = (data['close'] - data['close_1_month']) / data['close_1_month']
    data['return_3m'] = (data['close'] - data['close_3_month']) / data['close_3_month']
    data['return_6m'] = (data['close'] - data['close_6_month']) / data['close_6_month']
    data['return_12m'] = (data['close'] - data['close_12_month']) / data['close_12_month']
    data['return_avg'] = data[['return_1m', 'return_3m', 'return_6m', 'return_12m']].mean(axis=1)
    data["mom_score"] = (
        12 * data["return_1m"] +
        4 * data["return_3m"] +
        2 * data["return_6m"] +
        1 * data["return_12m"]
    ) / 19

    # 평균 return
    data['return_avg'] = data[['return_1m', 'return_3m', 'return_6m', 'return_12m']].mean(axis=1)
    
    # 모멘텀 점수 (mom_score)
    data['mom_score'] = (
        12 * data['return_1m'] +
        4 * data['return_3m'] +
        2 * data['return_6m'] +
        1 * data['return_12m']
    ) / 19
    
    return data


In [35]:
future_risk_df = calculate_momentum_features(future_risk_df)
index_risk_df = calculate_momentum_features(index_risk_df)
future_minvar_df = calculate_momentum_features(future_minvar_df)
index_minvar_df = calculate_momentum_features(index_minvar_df)
future_ms_df = calculate_momentum_features(future_ms_df)
index_ms_df = calculate_momentum_features(index_ms_df)
future_paa_df = calculate_momentum_features(future_paa_df)
index_paa_df = calculate_momentum_features(index_paa_df)

In [36]:
future_risk_df["ret"] = future_risk_df["close"].pct_change().shift(1)
index_risk_df["ret"] = index_risk_df["close"].pct_change().shift(1)
future_minvar_df["ret"] = future_minvar_df["close"].pct_change().shift(1)
index_minvar_df["ret"] = index_minvar_df["close"].pct_change().shift(1)
future_ms_df["ret"] = future_ms_df["close"].pct_change().shift(1)
index_ms_df["ret"] = index_ms_df["close"].pct_change().shift(1)
future_paa_df["ret"] = future_paa_df["close"].pct_change().shift(1)
index_paa_df["ret"] = index_paa_df["close"].pct_change().shift(1)

In [37]:
eval_feature = ['vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [38]:
future_risk_df = pd.concat([future_risk_df, future_rp_eval[eval_feature]], axis=1)
index_risk_df = pd.concat([index_risk_df, index_rp_eval[eval_feature]], axis=1)
future_minvar_df = pd.concat([future_minvar_df, future_minvar_eval[eval_feature]], axis=1)
index_minvar_df = pd.concat([index_minvar_df, index_minvar_eval[eval_feature]], axis=1)
future_ms_df = pd.concat([future_ms_df, future_ms[eval_feature]], axis=1)
index_ms_df = pd.concat([index_ms_df, index_ms[eval_feature]], axis=1)
future_paa_df = pd.concat([future_paa_df, future_paa[eval_feature]], axis=1)
index_paa_df = pd.concat([index_paa_df, index_paa[eval_feature]], axis=1)

In [39]:
# portfolio_df = pd.concat([future_risk_df, index_risk_df, future_ms_df, index_ms_df, future_paa_df, index_paa_df], axis=0)


In [40]:
portfolio_df = pd.concat([future_risk_df, index_risk_df, future_minvar_df, index_minvar_df, future_ms_df, index_ms_df, future_paa_df, index_paa_df], axis=0)

In [41]:
portfolio_df.reset_index(inplace=True)

In [42]:
portfolio_df.rename(columns={"index": "date"}, inplace=True)

In [43]:
portfolio_df.tic.unique()

array(['future_risk', 'index_risk', 'future_minvar', 'index_minvar',
       'future_ms', 'index_ms', 'future_paa', 'index_paa'], dtype=object)

In [44]:
portfolio_df.columns

Index(['date', 'close', 'tic', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score', 'ret', 'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252'],
      dtype='object')

In [45]:
select_features = ['return_1m',
       'return_3m', 'return_6m', 'return_12m', 'return_avg',
       'ret', 'vol_20', 'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [46]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp[select_features].describe())

future_risk
         return_1m    return_3m    return_6m   return_12m   return_avg  \
count  5711.000000  5671.000000  5611.000000  5479.000000  5711.000000   
mean      0.009332     0.028663     0.058895     0.126332     0.054892   
std       0.042876     0.081022     0.130288     0.207294     0.096178   
min      -0.272854    -0.355468    -0.435141    -0.391055    -0.273186   
25%      -0.016042    -0.014756    -0.022513    -0.027594    -0.010211   
50%       0.009003     0.027676     0.053003     0.105033     0.047912   
75%       0.036161     0.076724     0.134099     0.260464     0.110721   
max       0.201133     0.360045     0.532237     0.804622     0.426182   

               ret       vol_20   sharpe_252      vol_252  sortino_252  \
count  5729.000000  5711.000000  5479.000000  5479.000000  5479.000000   
mean      0.000459     0.132787     0.912211     0.139178     1.343019   
std       0.009148     0.058207     1.414107     0.045546     2.030703   
min      -0.056206     0.

In [47]:
portfolio_df.sort_values(by=["date", "tic"], inplace=True)

In [48]:
portfolio_df.reset_index(drop=True, inplace=True)

In [49]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp.describe())

future_minvar
             close  close_1_month  close_2_month  close_3_month  \
count  5731.000000    5711.000000    5691.000000    5671.000000   
mean      5.619168       5.587103       5.555534       5.523707   
std       3.164842       3.123554       3.083207       3.041608   
min       0.997528       0.997528       0.997528       0.997528   
25%       3.455711       3.440316       3.415409       3.403455   
50%       5.415541       5.412296       5.409983       5.404551   
75%       6.514908       6.500901       6.487101       6.469801   
max      15.030275      14.836168      14.749930      14.705311   

       close_4_month  close_5_month  close_6_month  close_7_month  \
count    5651.000000    5631.000000    5611.000000    5591.000000   
mean        5.492832       5.464275       5.435706       5.406657   
std         3.002198       2.968955       2.935326       2.900026   
min         0.997528       0.997528       0.997528       0.997528   
25%         3.382616       3.330143  

In [50]:
for i in portfolio_df.tic.unique():
    imp = portfolio_df[portfolio_df.tic == i].copy()
    print(i)
    print(imp.isna().sum())

future_minvar
date                0
close               0
tic                 0
close_1_month      20
close_2_month      40
close_3_month      60
close_4_month      80
close_5_month     100
close_6_month     120
close_7_month     140
close_8_month     160
close_9_month     180
close_10_month    220
close_11_month    240
close_12_month    252
return_1m          20
return_3m          60
return_6m         120
return_12m        252
return_avg         20
mom_score         252
ret                 2
vol_20             20
sharpe_252        252
vol_252           252
sortino_252       252
calmar_252        252
dtype: int64
future_ms
date                0
close               0
tic                 0
close_1_month      20
close_2_month      40
close_3_month      60
close_4_month      80
close_5_month     100
close_6_month     120
close_7_month     140
close_8_month     160
close_9_month     180
close_10_month    220
close_11_month    240
close_12_month    252
return_1m          20
return_3m        

In [51]:
portfolio_na = portfolio_df.dropna()

In [52]:
portfolio_na.reset_index(drop=True, inplace=True)

In [53]:
for tic  in portfolio_na.tic.unique():
    imp = portfolio_na[portfolio_na.tic == tic].copy()
    print(tic)
    print(imp.date.min())
    print(imp.date.max())

future_minvar
2003-02-19
2024-12-31
future_ms
2003-02-19
2024-12-31
future_paa
2003-02-19
2024-12-31
future_risk
2003-02-19
2024-12-31
index_minvar
2003-02-19
2024-12-31
index_ms
2003-02-19
2024-12-31
index_paa
2003-02-19
2024-12-31
index_risk
2003-02-19
2024-12-31


In [54]:
portfolio_na.rename(columns={"tic": "ticker"}, inplace=True)

/tmp/ipykernel_1679210/1574858568.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  portfolio_na.rename(columns={"tic": "ticker"}, inplace=True)


In [56]:
TRAIN_START_DATE = '2003-01-01'
TRAIN_END_DATE = '2013-12-31'
VAILD_START_DATE = '2014-01-01'
VAILD_END_DATE = '2018-12-31'
TEST_START_DATE = '2019-01-01'
TEST_END_DATE = '2024-12-31'

In [57]:
def data_split(df, start, end, target_date_col="date"):
    """
    split the dataset into training or testing using date
    :param data: (df) pandas dataframe, start, end
    :return: (df) pandas dataframe
    """
    data = df[(df[target_date_col] >= start) & (df[target_date_col] <= end)]
    data = data.sort_values([target_date_col, "ticker"], ignore_index=True)
    # data.index = data[target_date_col].factorize()[0]
    return data

In [58]:
train = data_split(portfolio_na, TRAIN_START_DATE,TRAIN_END_DATE)
vaild = data_split(portfolio_na, VAILD_START_DATE,VAILD_END_DATE)
test = data_split(portfolio_na, TEST_START_DATE,TEST_END_DATE)

In [56]:
# def min_max_normalize_by_ticker_train_test(train_df, vaild_df, test_df, columns):
#     """
#     주어진 컬럼들을 train 기준으로 티커별 min-max 정규화
    
#     Parameters:
#         train_df (DataFrame): 학습용 데이터
#         test_df (DataFrame): 테스트용 데이터
#         columns (list): 정규화할 컬럼 리스트
        
#     Returns:
#         train_df, test_df: 정규화된 결과가 포함된 데이터프레임
#     """
#     # 티커별로 정규화 통계 계산
#     stats = train_df.groupby("ticker")[columns].agg(["min", "max"])
    
#     # 컬럼명 정리 (MultiIndex → flat column name)
#     stats.columns = [f"{col}_{stat}" for col, stat in stats.columns]

#     # train/test에 붙이기
#     train_df = train_df.merge(stats, on="ticker", how="left")
#     vaild_df = vaild_df.merge(stats, on="ticker", how="left")
#     test_df = test_df.merge(stats, on="ticker", how="left")

#     # 컬럼별 정규화 수행
#     for col in columns:
#         min_col = f"{col}_min"
#         max_col = f"{col}_max"
#         norm_col = f"{col}_norm"

#         train_df[norm_col] = (train_df[col] - train_df[min_col]) / (train_df[max_col] - train_df[min_col])
#         vaild_df[norm_col] = (vaild_df[col] - vaild_df[min_col]) / (vaild_df[max_col] - vaild_df[min_col])
#         test_df[norm_col] = (test_df[col] - test_df[min_col]) / (test_df[max_col] - test_df[min_col])

#     # 불필요한 min/max 컬럼 제거
#     cols_to_drop = [f"{col}_min" for col in columns] + [f"{col}_max" for col in columns]
#     train_df.drop(columns=cols_to_drop, inplace=True)
#     vaild_df.drop(columns=cols_to_drop, inplace=True)
#     test_df.drop(columns=cols_to_drop, inplace=True)

#     return train_df, vaild_df, test_df


In [59]:
def min_max_normalize_global_train_test(train_df, valid_df, test_df, columns):
    """
    전체 train 데이터 기준으로 주어진 컬럼들에 대해 min-max 정규화
    
    Parameters:
        train_df (DataFrame): 학습용 데이터
        valid_df (DataFrame): 검증용 데이터
        test_df (DataFrame): 테스트용 데이터
        columns (list): 정규화할 컬럼 리스트
        
    Returns:
        train_df, valid_df, test_df: 정규화된 결과가 포함된 데이터프레임
    """
    # train 데이터 전체 기준으로 min/max 계산
    stats = train_df[columns].agg(["min", "max"])
    
    for col in columns:
        min_val = stats.loc["min", col]
        max_val = stats.loc["max", col]
        norm_col = f"{col}_norm"
        
        train_df[norm_col] = (train_df[col] - min_val) / (max_val - min_val)
        valid_df[norm_col] = (valid_df[col] - min_val) / (max_val - min_val)
        test_df[norm_col] = (test_df[col] - min_val) / (max_val - min_val)
    
    return train_df, valid_df, test_df


In [60]:
columns_to_normalize = ["ret", 'close', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score',  'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252']

In [61]:
# train_df, vaild_df, test_df = min_max_normalize_by_ticker_train_test(train, vaild, test, columns_to_normalize)

In [62]:
train_df, vaild_df, test_df = min_max_normalize_global_train_test(train, vaild, test, columns_to_normalize)

In [63]:
train_df[['ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm']].describe()

,ret_norm,close_norm,return_1m_norm,return_3m_norm,return_6m_norm,return_12m_norm,return_avg_norm,mom_score_norm,vol_20_norm,sharpe_252_norm,vol_252_norm,sortino_252_norm
count,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000,21744.000000
mean,0.485824,0.249325,0.591388,0.479497,0.452181,0.483371,0.543241,0.569462,0.203859,0.405401,0.380090,0.308708
std,0.062218,0.195542,0.089181,0.100409,0.116707,0.159397,0.113967,0.091569,0.105071,0.162066,0.180548,0.142241
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.456478,0.124116,0.542882,0.423070,0.385892,0.370446,0.470387,0.516920,0.132782,0.274534,0.244630,0.192110
50%,0.486512,0.183063,0.593835,0.479441,0.447031,0.496384,0.546112,0.570853,0.182129,0.413551,0.344709,0.308847
75%,0.518297,0.305869,0.645112,0.541686,0.524210,0.583268,0.614492,0.624943,0.253830,0.517642,0.494904,0.402110
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [64]:
for tic in test_df.ticker.unique():
    imp = train_df[train_df.ticker == tic].copy()
    print(tic)
    print(imp.describe())


future_minvar
             close  close_1_month  close_2_month  close_3_month  \
count  2718.000000    2718.000000    2718.000000    2718.000000   
mean      3.969472       3.937897       3.906420       3.873495   
std       1.898156       1.906276       1.914840       1.922406   
min       1.238005       1.238005       1.238005       1.187582   
25%       2.017592       1.982039       1.900517       1.866893   
50%       3.803515       3.769687       3.729199       3.703729   
75%       5.746971       5.746971       5.746971       5.740709   
max       7.797319       7.797319       7.797319       7.797319   

       close_4_month  close_5_month  close_6_month  close_7_month  \
count    2718.000000    2718.000000    2718.000000    2718.000000   
mean        3.840580       3.807306       3.774271       3.739955   
std         1.929777       1.936653       1.943386       1.948911   
min         1.174076       1.153118       1.119525       1.085050   
25%         1.847392       1.822278  

In [65]:
train_df.to_csv("../data/portfolio_price/train_portfolio_price_v6.csv", index=False)
vaild_df.to_csv("../data/portfolio_price/vaild_portfolio_price_v6.csv", index=False)
test_df.to_csv("../data/portfolio_price/test_portfolio_price_v6.csv", index=False)

In [66]:
train_df.columns

Index(['date', 'close', 'ticker', 'close_1_month', 'close_2_month',
       'close_3_month', 'close_4_month', 'close_5_month', 'close_6_month',
       'close_7_month', 'close_8_month', 'close_9_month', 'close_10_month',
       'close_11_month', 'close_12_month', 'return_1m', 'return_3m',
       'return_6m', 'return_12m', 'return_avg', 'mom_score', 'ret', 'vol_20',
       'sharpe_252', 'vol_252', 'sortino_252', 'calmar_252', 'ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm',
       'calmar_252_norm'],
      dtype='object')

In [67]:
select_features = ['ret_norm',
       'close_norm', 'return_1m_norm', 'return_3m_norm', 'return_6m_norm',
       'return_12m_norm', 'return_avg_norm', 'mom_score_norm', 'vol_20_norm',
       'sharpe_252_norm', 'vol_252_norm', 'sortino_252_norm',
       'calmar_252_norm']

In [68]:
tot_data = pd.concat([train_df, vaild_df, test_df])

In [69]:
tot_data = tot_data.sort_values(["date", "ticker"])

In [70]:
describe_df = tot_data[select_features].describe(include='all')

In [71]:
describe_df = vaild_df[select_features].describe(include='all')

In [72]:
# 고유 날짜/종목 추출 및 정렬
dates = np.sort(tot_data['date'].unique())
tics = np.sort(tot_data['ticker'].unique())

In [73]:
# 인덱스 매핑 (빠른 접근용)
date2idx = {d: i for i, d in enumerate(dates)}
tic2idx = {t: i for i, t in enumerate(tics)}

In [74]:
state_array = np.zeros((len(dates), len(tics), len(select_features)), dtype=np.float32)

In [75]:
for row in tot_data.itertuples():
    d_idx = date2idx[row.date]
    t_idx = tic2idx[row.ticker]
    f_vals = [getattr(row, f) for f in select_features]
    state_array[d_idx, t_idx, :] = np.nan_to_num(f_vals)  # NaN은 0으로

In [76]:
state_array[0]

array([[0.4932013 , 0.07045679, 0.5835025 , 0.5424838 , 0.52593315,
        0.590786  , 0.6230106 , 0.6081516 , 0.1286693 , 0.67270714,
        0.15950008, 0.52856094, 0.31062186],
       [0.5284082 , 0.07605288, 0.6209084 , 0.57130545, 0.48307452,
        0.63833696, 0.6390979 , 0.6325166 , 0.16274726, 0.5544226 ,
        0.32309353, 0.40891016, 0.19874266],
       [0.59149843, 0.08032509, 0.73081666, 0.67502683, 0.55294466,
        0.6458035 , 0.70964366, 0.7331518 , 0.28614292, 0.5158814 ,
        0.42060003, 0.4158895 , 0.19533342],
       [0.5028176 , 0.06872519, 0.5958774 , 0.54830164, 0.51343095,
        0.582907  , 0.6192565 , 0.61307037, 0.13353744, 0.6480036 ,
        0.15800855, 0.50461817, 0.29601052],
       [0.48897317, 0.0232264 , 0.5459378 , 0.480121  , 0.37642443,
        0.37150222, 0.4690983 , 0.5202306 , 0.09618604, 0.24536352,
        0.23369484, 0.16600695, 0.04209929],
       [0.44473192, 0.00993822, 0.5817971 , 0.50719976, 0.33502525,
        0.297472  , 0.44020

In [77]:
tot_data[tot_data["date"]=="2018-12-31"][select_features]

,ret_norm,close_norm,return_1m_norm,return_3m_norm,return_6m_norm,return_12m_norm,return_avg_norm,mom_score_norm,vol_20_norm,sharpe_252_norm,vol_252_norm,sortino_252_norm,calmar_252_norm
10000,0.480890,0.664128,0.579720,0.479805,0.418980,0.373149,0.484128,0.558900,0.112273,0.250433,0.151150,0.173324,0.055177
10001,0.246343,-0.014544,0.317406,0.151110,0.163242,0.142140,0.185433,0.250279,0.512042,0.085497,0.478544,0.059285,0.027443
10002,-0.045902,0.038430,0.177215,0.149296,0.162621,0.188086,0.178427,0.176868,1.070196,0.167152,0.738172,0.126318,0.032282
10003,0.486706,0.512526,0.552986,0.432226,0.388590,0.360819,0.452805,0.523826,0.137496,0.231349,0.182995,0.159029,0.047872
10004,0.508234,0.215049,0.468480,0.327508,0.358442,0.320287,0.386922,0.437872,0.234188,0.165040,0.207593,0.117371,0.041404
10005,0.465602,0.176758,0.419047,0.287950,0.336594,0.274243,0.344227,0.390640,0.419454,0.141550,0.302260,0.099330,0.038482
10006,0.472517,0.186340,0.538842,0.350307,0.350315,0.276293,0.385436,0.477180,0.000000,0.097728,0.207201,0.071518,0.034929
10007,0.547952,0.170008,0.463444,0.315170,0.324739,0.286404,0.359101,0.421903,0.256532,0.129378,0.250181,0.088288,0.036118


In [77]:
TRAIN_START_DATE = '2003-01-01'
TRAIN_END_DATE = '2013-12-31'
VAILD_START_DATE = '2014-01-01'
VAILD_END_DATE = '2018-12-31'
TEST_START_DATE = '2019-01-01'
TEST_END_DATE = '2024-12-31'

In [78]:
valid_start_date = "2011-01-03"
test_start_date = "2017-01-03"

In [79]:
date2idx['2014-01-02']

2718

In [80]:
train = state_array[:2718]

In [81]:
valid = state_array[2718:3969]

In [82]:
test = state_array[3969:]

In [83]:
import torch

In [84]:
train_tensor = torch.from_numpy(train)
valid_tensor = torch.from_numpy(valid)
test_tensor = torch.from_numpy(test)

In [85]:
train.shape

(2718, 8, 13)

In [86]:
train_tensor.shape

torch.Size([2718, 8, 13])

In [87]:
valid_tensor.shape

torch.Size([1251, 8, 13])

In [88]:
torch.save(train_tensor, "../data/portfolio_price/portfolio_train_v5.pt")
torch.save(test_tensor,  "../data/portfolio_price/portfolio_test_v5.pt")
torch.save(valid_tensor, "../data/portfolio_price/portfolio_valid_v5.pt")

In [89]:
def calculate_annual_return(returns, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    cumulative = np.prod(1 + returns)
    n_periods = len(returns)
    return cumulative ** (annual_factor / n_periods) - 1


def calculate_max_drawdown(returns):
    returns = np.array(returns, dtype=np.float64)
    cumulative_returns = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdowns = (cumulative_returns - running_max) / running_max
    return abs(np.min(drawdowns))

def calculate_volatility(returns, annual_factor=252):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1) * np.sqrt(annual_factor)

def calculate_sharpe_ratio(returns, risk_free_rate=0.02, annual_factor=252):
    returns = np.array(returns, dtype=np.float64)
    annual_return = calculate_annual_return(returns, annual_factor)
    annual_std_dev = calculate_volatility(returns, annual_factor)
    excess_return =  (annual_return - risk_free_rate) 
    # 연환산 수익률과 표준편차
    return excess_return / annual_std_dev if annual_std_dev != 0 else 0.0


def calculate_cumulative_return(returns):
    returns = np.array(returns, dtype=np.float64)
    return np.prod(1 + returns) - 1

def calculate_volatility(returns, annual_factor=252):
    returns = np.asarray(returns)
    return np.std(returns, ddof=1) * np.sqrt(annual_factor)


# 성과 지표 계산 함수 (파라미터화)
def calculate_performance_metrics(returns, annual_factor=252, risk_free_rate=0.0):
    cumulative_returns = (1 + returns).cumprod()
    # cagr = calculate_cagr(returns, annual_factor)
    annual_return = calculate_annual_return(returns, annual_factor)
    sharpe_ratio = calculate_sharpe_ratio(returns, risk_free_rate, annual_factor)
    mdd = calculate_max_drawdown(returns)
    volatility = calculate_volatility(returns, annual_factor)

    return {
        'Cumulative Return': cumulative_returns.iloc[-1] - 1,
        'Annual Return': annual_return,
        # 'CAGR': cagr,
        'Volatility': volatility,
        'Sharpe Ratio': sharpe_ratio,
        'MDD': mdd
    }


# 성과 지표 요약 생성 함수 (파라미터 전달)
def get_performance_summary(returns_dict, annual_factor=252, risk_free_rate=0.0):
    metrics_df = pd.DataFrame()
    for name, returns in returns_dict.items():
        metrics_df[name] = calculate_performance_metrics(returns, annual_factor, risk_free_rate)
    return metrics_df.T


In [90]:
metrics_summary = get_performance_summary(
    strategy_returns,
    annual_factor=252,
    risk_free_rate=0.0
)

display(metrics_summary)


,Cumulative Return,Annual Return,Volatility,Sharpe Ratio,MDD
future_risk_parity,10.044736,0.111396,0.145215,0.767116,0.472598
index_risk_parity,1.608454,0.043059,0.158647,0.271415,0.616640
future_minvar,13.846313,0.125946,0.131101,0.960683,0.395177
index_minvar,1.446192,0.040118,0.118418,0.338780,0.492377
future_ms,2.008415,0.049623,0.213534,0.232387,0.786581
index_ms,0.659116,0.022512,0.134920,0.166852,0.551940
future_paa,5.889434,0.088569,0.213662,0.414529,0.621110
index_paa,2.475320,0.056302,0.145651,0.386557,0.373808
Equal Weight,3.818659,0.071591,0.148739,0.481322,0.521208


In [91]:
strategy_returns

{'future_risk_parity': 2002-02-05    0.010707
 2002-02-06   -0.001129
 2002-02-07   -0.000104
 2002-02-08    0.009986
 2002-02-11   -0.006430
                 ...   
 2024-12-24    0.002051
 2024-12-26   -0.000356
 2024-12-27   -0.001801
 2024-12-30    0.000730
 2024-12-31    0.003271
 Length: 5731, dtype: float64,
 'index_risk_parity': 2002-02-05   -0.015220
 2002-02-06   -0.005098
 2002-02-07    0.004381
 2002-02-08    0.004523
 2002-02-11    0.010565
                 ...   
 2024-12-24    0.004831
 2024-12-26    0.002980
 2024-12-27   -0.002778
 2024-12-30   -0.003768
 2024-12-31   -0.000138
 Length: 5731, dtype: float64,
 'future_minvar': 2002-02-05    0.011503
 2002-02-06    0.000010
 2002-02-07   -0.001303
 2002-02-08    0.008598
 2002-02-11   -0.008424
                 ...   
 2024-12-24    0.001171
 2024-12-26    0.001245
 2024-12-27   -0.003861
 2024-12-30    0.001253
 2024-12-31    0.007062
 Length: 5731, dtype: float64,
 'index_minvar': 2002-02-05   -0.010447
 2002-02-06   -